## **LangChain ecosystem**

**LangChain is an open-source framework for building custom agents and applications powered by LLMs in under 10 lines of code.**


**What it does:**
- Connects to OpenAI, Anthropic, Google, and other LLM providers
- Provides pre-built agent architecture and integrations
- Standardizes model interfaces across different providers

**When to use:**
- **LangChain** → Quick agent building with flexibility
- **Deep Agents** → If you need modern features (conversation compression, virtual filesystem, subagents) - recommended starting point
- **LangGraph** → For advanced customization, deterministic + agentic workflows

**Core benefits:**
- Standard model interface (swap providers without lock-in)
- Simple yet flexible agent abstraction
- Built on LangGraph (durable execution, persistence, human-in-the-loop)
- Debug with LangSmith visualization


In [1]:
print("Langchain. Learn with ❤️")

Langchain. Learn with ❤️


In [2]:
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

model = init_chat_model(model="moonshotai/kimi-k2-instruct-0905", model_provider="groq")

def get_weather(city: str) -> str:
    """Get the weather of the city"""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

response = agent.invoke({"messages": [{"role": "user", "content": "what is the weather in sf"}]})

In [3]:
from pprint import pprint

print("type(response):", type(response))
print("top-level keys:", list(response.keys()))
pprint(response)


type(response): <class 'dict'>
top-level keys: ['messages']
{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='f89dabda-013d-484b-92b7-525b68d660ca'),
              AIMessage(content="I'll get the current weather for San Francisco.", additional_kwargs={'tool_calls': [{'id': 'functions.get_weather:0', 'function': {'arguments': '{"city":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 86, 'total_tokens': 114, 'completion_time': 0.107681625, 'completion_tokens_details': None, 'prompt_time': 0.013786775, 'prompt_tokens_details': None, 'queue_time': 0.276093114, 'total_time': 0.1214684}, 'model_name': 'moonshotai/kimi-k2-instruct-0905', 'system_fingerprint': 'fp_00c37775b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ca2bd-a89b-7ef2-8273-660e352ee699-0', tool_cal

In [4]:
import json

def pretty_agent_response(response: dict):
    msgs = response.get("messages", [])
    print(f"Total messages: {len(msgs)}\n")

    for i, m in enumerate(msgs, 1):
        role = type(m).__name__
        print(f"[{i}] {role}")

        # Main text
        content = getattr(m, "content", "")
        if content:
            print(f"  content: {content}")

        # Tool calls (if any)
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            print("  tool_calls:")
            for tc in tool_calls:
                print(f"    - {tc.get('name')}({tc.get('args')}) id={tc.get('id')}")

        # Tool message metadata
        tool_name = getattr(m, "name", None)
        tool_call_id = getattr(m, "tool_call_id", None)
        if tool_name or tool_call_id:
            print(f"  tool: name={tool_name}, tool_call_id={tool_call_id}")

        # Token usage (if available)
        usage = getattr(m, "usage_metadata", None) or {}
        if usage:
            print(
                f"  tokens: in={usage.get('input_tokens')}, "
                f"out={usage.get('output_tokens')}, total={usage.get('total_tokens')}"
            )

        print()  # blank line between messages

pretty_agent_response(response)


Total messages: 4

[1] HumanMessage
  content: what is the weather in sf

[2] AIMessage
  content: I'll get the current weather for San Francisco.
  tool_calls:
    - get_weather({'city': 'San Francisco'}) id=functions.get_weather:0
  tokens: in=86, out=28, total=114

[3] ToolMessage
  content: It's always sunny in San Francisco!
  tool: name=get_weather, tool_call_id=functions.get_weather:0

[4] AIMessage
  content: According to the weather service, it's always sunny in San Francisco!
  tokens: in=138, out=14, total=152



In [5]:
def inspect_obj(obj, name="obj", depth=0, max_depth=4):
    indent = "  " * depth
    print(f"{indent}{name}: {type(obj)}")

    if depth >= max_depth:
        return

    if isinstance(obj, dict):
        for k, v in obj.items():
            inspect_obj(v, f"{name}[{k!r}]", depth + 1, max_depth)
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            inspect_obj(v, f"{name}[{i}]", depth + 1, max_depth)
    else:
        attrs = [a for a in dir(obj) if not a.startswith("_")]
        if attrs:
            print(f"{indent}  attrs: {attrs}")

inspect_obj(response, "response")


response: <class 'dict'>
  response['messages']: <class 'list'>
    response['messages'][0]: <class 'langchain_core.messages.human.HumanMessage'>
      attrs: ['additional_kwargs', 'construct', 'content', 'content_blocks', 'copy', 'dict', 'from_orm', 'get_lc_namespace', 'id', 'is_lc_serializable', 'json', 'lc_attributes', 'lc_id', 'lc_secrets', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'name', 'parse_file', 'parse_obj', 'parse_raw', 'pretty_print', 'pretty_repr', 'response_metadata', 'schema', 'schema_json', 'text', 'to_json', 'to_json_not_implemented', 'type', 'update_forward_refs', 'validate']
    response['messages'][1]: <class 'langchain_core.messages.ai.AIMessage'>
      attrs: ['additional_kwargs', 'construct', 'conte

In [6]:
last_msg = response["messages"][-1]
print(type(last_msg))
print(last_msg)               # readable repr
print(last_msg.content)       # final text
print(last_msg.__dict__)      # works if this is an object with __dict__
# or, for pydantic-like objects:
# print(last_msg.model_dump())


<class 'langchain_core.messages.ai.AIMessage'>
content="According to the weather service, it's always sunny in San Francisco!" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 138, 'total_tokens': 152, 'completion_time': 0.024991618, 'completion_tokens_details': None, 'prompt_time': 0.016753701, 'prompt_tokens_details': None, 'queue_time': 0.279446089, 'total_time': 0.041745319}, 'model_name': 'moonshotai/kimi-k2-instruct-0905', 'system_fingerprint': 'fp_00c37775b7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ca2bd-aaa4-7690-b0a9-fef9d9fe36fb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 138, 'output_tokens': 14, 'total_tokens': 152}
According to the weather service, it's always sunny in San Francisco!
{'content': "According to the weather service, it's always sunny in San Francisco!", 'additional_kwargs': {}, 'response_metadata': {'token_usage

### **Build a real-world agent**

Next, build a practical weather forecasting agent that demonstrates key production concepts:

1. **Detailed system prompts** for better agent behavior
2. **Create tools** that integrate with external data
3. **Model configuration** for consistent responses
4. **Structured output** for predictable results
5. **Conversational memory** for chat-like interactions
6. **Create and run the agent** to test the fully functional agent


In [7]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

In [8]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime, tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

@tool
def get_weather_for_location(city: str) ->str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"


model = init_chat_model(model="moonshotai/kimi-k2-instruct-0905", model_provider="groq")

# Define response format
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [9]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [10]:
# Create and run the agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.runnables import RunnableConfig

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_weather_for_location, get_user_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [
            {"role": "user", "content": "what is the weather outside?"}
    ]},
    config=config,
    context=Context(user_id="1")
)

In [11]:
print(response['structured_response'])

ResponseFormat(punny_response="Well, looks like Florida's living up to its sunshine state reputation - it's always sunny there! You could say the weather is absolutely *Florid-awesome*! No clouds raining on your parade today - it's a perfect day to soak up those rays and work on your *sun*-sational tan!", weather_conditions="It's always sunny in Florida!")


In [15]:
last_msg = response["messages"][-1]
# print(last_msg)               # readable repr
print(last_msg.content)       # final text
# print(last_msg.__dict__)      # works if this is an object with __dict__
# print(last_msg.model_dump())


Returning structured response: ResponseFormat(punny_response="Well, looks like Florida's living up to its sunshine state reputation - it's always sunny there! You could say the weather is absolutely *Florid-awesome*! No clouds raining on your parade today - it's a perfect day to soak up those rays and work on your *sun*-sational tan!", weather_conditions="It's always sunny in Florida!")


In [16]:
pretty_agent_response(response)

Total messages: 7

[1] HumanMessage
  content: what is the weather outside?

[2] AIMessage
  tool_calls:
    - get_user_location({}) id=functions.get_user_location:0
  tokens: in=302, out=25, total=327

[3] ToolMessage
  content: Florida
  tool: name=get_user_location, tool_call_id=functions.get_user_location:0

[4] AIMessage
  tool_calls:
    - get_weather_for_location({'city': 'Florida'}) id=functions.get_weather_for_location:1
  tokens: in=333, out=20, total=353

[5] ToolMessage
  content: It's always sunny in Florida!
  tool: name=get_weather_for_location, tool_call_id=functions.get_weather_for_location:1

[6] AIMessage
  tool_calls:
    - ResponseFormat({'punny_response': "Well, looks like Florida's living up to its sunshine state reputation - it's always sunny there! You could say the weather is absolutely *Florid-awesome*! No clouds raining on your parade today - it's a perfect day to soak up those rays and work on your *sun*-sational tan!", 'weather_conditions': "It's always su

In [19]:
# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in New York?"}]},
    config=config,
    context=Context(user_id="1")
)


In [23]:
print(response['structured_response'].punny_response)

Looks like the Big Apple is serving up some *sun*sational weather! New York's got that perpetual sunshine - I guess you could say it's a *central park*-ing lot of sunshine up there! No need to feel *brooklynn* blue about the weather today - it's absolutely *times square*-tacular!
